In [ ]:
# ============================================================
# demo.ipynb — OLMo2-1B 투명성 실험 전체 데모
# 논문: OLMo: Accelerating the Science of Language Models
# 실험 4개: 투명성 확인 / 패턴 재현 / LoRA SFT / 도메인 확장
# 런타임: RTX 4090 권장
# ============================================================

# 0. 패키지 설치
# pip install transformers peft trl==1.7.1 accelerate datasets
# pip install bitsandbytes huggingface_hub pandas

import torch
import pandas as pd
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from huggingface_hub import list_repo_refs, login

# HuggingFace 로그인
login(token='hf_여기에토큰입력')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# ============================================================
# PART 1. 투명성 확인 — 공개 체크포인트 목록
# 논문 근거 1: 연구 가능성
# ============================================================
print('\n' + '='*60)
print('PART 1. 투명성 확인: 공개 체크포인트 목록')
print('='*60)

out = list_repo_refs("allenai/OLMo-2-0425-1B")
branches = [b.name for b in out.branches]
print(f'총 체크포인트 수: {len(branches)}개')
print('예시 (최근 10개):')
for b in branches[:10]:
    print(f'  {b}')
print('\n→ 공개된 체크포인트가 있기 때문에 아래 실험 전체가 가능')
print('→ 닫힌 모델에서는 이 목록 자체가 존재하지 않음')

# ============================================================
# PART 2. Base → SFT → Instruct 패턴 비교
# 논문 근거 2: 재현 가능성 / 근거 3: 인과 추적
# ============================================================
print('\n' + '='*60)
print('PART 2. Base → SFT → Instruct 단계별 패턴 비교')
print('='*60)

# 영어 4태스크 프롬프트 로드
with open('./data/sample_cases.jsonl', encoding='utf-8') as f:
    cases = [json.loads(line) for line in f]

TASKS_EN = {}
for c in cases:
    TASKS_EN.setdefault(c['task'], []).append(c['prompt'])

MODEL_IDS = {
    'Base'    : 'allenai/OLMo-2-0425-1B',
    'SFT'     : 'allenai/OLMo-2-0425-1B-SFT',
    'Instruct': 'allenai/OLMo-2-0425-1B-Instruct'
}
STAGE_ORDER = ['Base', 'SFT', 'Instruct']


def build_prompt(tokenizer, stage, text):
    if stage == 'Base':
        return text
    if tokenizer.chat_template is None:
        return f"### Instruction:\n{text}\n\n### Response:\n"
    try:
        return tokenizer.apply_chat_template(
            [{'role': 'user', 'content': text}],
            tokenize=False, add_generation_prompt=True
        )
    except Exception:
        return f"### Instruction:\n{text}\n\n### Response:\n"


def generate(tokenizer, model, stage, text, max_new_tokens=150):
    prompt = build_prompt(tokenizer, stage, text)
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()


all_rows = []

for stage, model_id in MODEL_IDS.items():
    print(f'\n[로딩] {stage}: {model_id}')
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, dtype=torch.float16, device_map='auto'
    )
    model.eval()

    for task, prompts in TASKS_EN.items():
        for i, p in enumerate(prompts):
            resp = generate(tokenizer, model, stage, p)
            all_rows.append({
                'stage': stage, 'task': task,
                'idx': i+1, 'prompt': p,
                'response': resp, 'resp_len': len(resp)
            })
            print(f'  [{task}] {i+1} 완료')

    del model
    torch.cuda.empty_cache()
    print(f'[{stage}] 완료 + 메모리 해제')

df_en = pd.DataFrame(all_rows)

for task in TASKS_EN:
    print(f'\n{"="*60}\n  Task: {task}\n{"="*60}')
    sub = df_en[df_en['task'] == task]
    for idx in sub['idx'].unique():
        p = sub[sub['idx']==idx].iloc[0]['prompt']
        print(f'\n  [Q{idx}] {p[:80]}')
        for stage in STAGE_ORDER:
            r = sub[(sub['idx']==idx) & (sub['stage']==stage)]
            if len(r):
                resp = r.iloc[0]['response']
                print(f'  [{stage:8s}] {resp[:120]}{"..." if len(resp)>120 else ""}')

print('\n=== 평균 응답 길이 (문자 수) ===')
print(df_en.groupby(['stage','task'])['resp_len'].mean().unstack().reindex(STAGE_ORDER).round(1))
print('\n=== 단계별 전체 평균 ===')
print(df_en.groupby('stage')['resp_len'].mean().reindex(STAGE_ORDER).round(1))

df_en.to_csv('./results/olmo2_english_results.csv', index=False, encoding='utf-8-sig')
print('\nCSV 저장 완료: results/olmo2_english_results.csv')

# ============================================================
# PART 3. LoRA SFT — TÜLU v2로 논문 SFT 경량 재현
# 논문 근거 2: 재현 가능성
# ============================================================
print('\n' + '='*60)
print('PART 3. LoRA 파인튜닝 — 논문 SFT 경량 재현')
print('='*60)

BASE_ID = 'allenai/OLMo-2-0425-1B'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(BASE_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{% if message['role'] == 'user' %}"
    "### Instruction:\n{{ message['content'] }}\n\n"
    "{% elif message['role'] == 'assistant' %}"
    "### Response:\n{{ message['content'] }}\n\n"
    "{% endif %}"
    "{% endfor %}"
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_ID, quantization_config=bnb_config, device_map='auto'
)
model.config.use_cache = False

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=16,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05, bias='none'
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# → trainable params: ~1M / 1.4B = 0.07%
# → 공개 가중치 없으면 이 줄 자체가 불가능

dataset = load_dataset(
    'allenai/tulu-v2-sft-mixture',
    split='train[:10000]',
    trust_remote_code=True
)
print(f'TÜLU v2 데이터 수: {len(dataset)}')


def format_tulu(example):
    messages = example.get('messages', [])
    text = ''
    for m in messages:
        if m.get('role') == 'user':
            text += f"### Instruction:\n{m.get('content','')}\n\n"
        elif m.get('role') == 'assistant':
            text += f"### Response:\n{m.get('content','')}\n\n"
    return {'text': text.strip()}


dataset = dataset.map(format_tulu)
dataset = dataset.train_test_split(test_size=0.02, seed=42)
print(f'train: {len(dataset["train"])} / test: {len(dataset["test"])}')

sft_config = SFTConfig(
    output_dir='./olmo2_tulu_sft',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    bf16=True,
    logging_steps=50,
    save_steps=200,
    eval_strategy='steps',
    eval_steps=200,
    dataset_text_field='text',
    report_to='none',
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    args=sft_config,
    processing_class=tokenizer,
)

print('\n학습 시작 (논문 동일 데이터 + LoRA 경량화)')
trainer.train()
trainer.save_model('./olmo2_tulu_sft_final')
tokenizer.save_pretrained('./olmo2_tulu_sft_final')
print('저장 완료: ./olmo2_tulu_sft_final')

# PART 3-2. 우리 SFT vs 공식 SFT 비교
print('\n=== 우리 SFT vs 공식 SFT 비교 ===')

TEST = [
    'What is the capital of France?',
    'Why should you not use elevators during a fire?',
    'Summarize in one sentence: AI mimics human intelligence through machine learning.'
]

results_compare = []

print('\n[우리 SFT 추론]')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_ID, dtype=torch.float16, device_map='auto'
)
our_sft = PeftModel.from_pretrained(base_model, './olmo2_tulu_sft_final')
our_sft.eval()

for p in TEST:
    prompt = f"### Instruction:\n{p}\n\n### Response:\n"
    inp = tokenizer(prompt, return_tensors='pt').to(device)
    ilen = inp['input_ids'].shape[1]
    with torch.no_grad():
        out = our_sft.generate(
            **inp, max_new_tokens=100,
            do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
    resp = tokenizer.decode(out[0][ilen:], skip_special_tokens=True).strip()
    results_compare.append({'prompt': p, 'our_sft': resp})
    print(f'  완료: {p[:50]}')

del our_sft, base_model
torch.cuda.empty_cache()

print('\n[공식 SFT 추론]')
off_tok = AutoTokenizer.from_pretrained('allenai/OLMo-2-0425-1B-SFT')
off_model = AutoModelForCausalLM.from_pretrained(
    'allenai/OLMo-2-0425-1B-SFT',
    dtype=torch.float16, device_map='auto'
)
off_model.eval()

for i, p in enumerate(TEST):
    if off_tok.chat_template:
        prompt = off_tok.apply_chat_template(
            [{'role': 'user', 'content': p}],
            tokenize=False, add_generation_prompt=True
        )
    else:
        prompt = f"### Instruction:\n{p}\n\n### Response:\n"
    inp = off_tok(prompt, return_tensors='pt').to(device)
    ilen = inp['input_ids'].shape[1]
    with torch.no_grad():
        out = off_model.generate(
            **inp, max_new_tokens=100,
            do_sample=False, pad_token_id=off_tok.eos_token_id
        )
    resp = off_tok.decode(out[0][ilen:], skip_special_tokens=True).strip()
    results_compare[i]['official_sft'] = resp
    print(f'  완료: {p[:50]}')

del off_model
torch.cuda.empty_cache()

print('\n' + '='*60)
for r in results_compare:
    print(f'\n[Q] {r["prompt"]}')
    print(f'[우리SFT  ] {r["our_sft"][:120]}')
    print(f'[공식SFT  ] {r.get("official_sft","")[:120]}')

pd.DataFrame(results_compare).to_csv(
    './results/olmo2_sft_comparison.csv', index=False, encoding='utf-8-sig'
)
print('\nCSV 저장 완료: results/olmo2_sft_comparison.csv')

# ============================================================
# PART 4. 도메인 확장 — 발달장애 직장 데이터 LoRA SFT+DPO
# 논문 근거 2: 재현 가능성 / 근거 3: 인과 추적
# ============================================================
print('\n' + '='*60)
print('PART 4. 도메인 비교 — 5개 모델 발달장애 직장 프롬프트')
print('='*60)

DOMAIN_PROMPTS = [
    "How should a workplace supervisor support an employee with developmental disabilities during onboarding?",
    "A worker with a developmental disability has difficulty matching the expected work pace. How should a support person help?",
    "How should a manager speak and act when a worker with a developmental disability cannot ask for help?",
    "What safety rules should be explained to a worker with a developmental disability at a new workplace?"
]

COMPARE_MODELS = {
    'Base'        : ('hf',   'allenai/OLMo-2-0425-1B'),
    'Our_SFT'     : ('lora', './olmo2_tulu_sft_final'),
    'Domain_SFT'  : ('lora', './domain_sft_clean'),
    'Domain_DPO'  : ('lora', './domain_dpo_clean'),
    'Official_SFT': ('hf',   'allenai/OLMo-2-0425-1B-SFT'),
}

results_domain = {name: [] for name in COMPARE_MODELS}

for name, (mtype, path) in COMPARE_MODELS.items():
    print(f'\n[로딩] {name}')

    if mtype == 'lora':
        base = AutoModelForCausalLM.from_pretrained(
            BASE_ID, dtype=torch.float16, device_map='auto'
        )
        m = PeftModel.from_pretrained(base, path)
        tok = AutoTokenizer.from_pretrained(BASE_ID)
    else:
        m = AutoModelForCausalLM.from_pretrained(
            path, dtype=torch.float16, device_map='auto'
        )
        tok = AutoTokenizer.from_pretrained(path)

    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    m.eval()

    for p in DOMAIN_PROMPTS:
        if name == 'Base':
            prompt = p
        elif mtype == 'lora':
            prompt = f"### Instruction:\n{p}\n\n### Response:\n"
        else:
            if tok.chat_template:
                prompt = tok.apply_chat_template(
                    [{'role': 'user', 'content': p}],
                    tokenize=False, add_generation_prompt=True
                )
            else:
                prompt = f"### Instruction:\n{p}\n\n### Response:\n"

        inp = tok(prompt, return_tensors='pt').to(device)
        ilen = inp['input_ids'].shape[1]
        with torch.no_grad():
            out = m.generate(
                **inp, max_new_tokens=200,
                do_sample=False, pad_token_id=tok.eos_token_id
            )
        resp = tok.decode(out[0][ilen:], skip_special_tokens=True).strip()
        results_domain[name].append(resp)
        print(f'  완료: {p[:50]}')

    del m
    if mtype == 'lora':
        del base
    torch.cuda.empty_cache()
    print(f'[{name}] 완료 + 메모리 해제')

print('\n' + '='*65)
print('PART 4. 5개 모델 도메인 프롬프트 비교')
print('='*65)

for i, p in enumerate(DOMAIN_PROMPTS):
    print(f'\n[Q{i+1}] {p}')
    print('-'*65)
    for name in COMPARE_MODELS:
        resp = results_domain[name][i]
        print(f'[{name:13s}] {resp[:150]}{"..." if len(resp)>150 else ""}')

rows = []
for i, p in enumerate(DOMAIN_PROMPTS):
    for name in COMPARE_MODELS:
        rows.append({
            'prompt_idx': i+1,
            'prompt'    : p,
            'model'     : name,
            'response'  : results_domain[name][i],
            'resp_len'  : len(results_domain[name][i])
        })

df_domain = pd.DataFrame(rows)
df_domain.to_csv(
    './results/olmo2_domain_comparison.csv',
    index=False, encoding='utf-8-sig'
)
print('\nCSV 저장 완료: results/olmo2_domain_comparison.csv')

print('\n=== 모델별 평균 응답 길이 ===')
print(df_domain.groupby('model')['resp_len'].mean().round(1))

print('\n' + '='*65)
print('최종 결론')
print('='*65)
print('논문 주장: 투명성이 과학적 연구를 가능하게 한다')
print()
print('실험 1: 체크포인트 268개 공개 확인    → 연구 가능성 입증')
print('실험 2: Base→SFT→Instruct 패턴 재현  → 재현 가능성 + 인과 추적')
print('실험 3: TÜLU LoRA SFT 재현           → 재현 가능성 심화')
print('실험 4: 발달장애 도메인 SFT+DPO      → 도메인 확장 가능성')
print()
print('→ 공개 가중치 + 공개 데이터 덕분에 전 단계 추적 및 확장 가능')
print('→ 닫힌 모델에서는 이 실험 전체가 불가능하다')